# Pseudobulk model building — PCA, NMF, ICA

**Environment:** `clamp-analyses`

Runs PCA, NMF, and ICA on every pseudobulk dataset. Reads the z-scored expression matrix (`norm.csv`) and rank estimate (`k.csv`) written by `01_CLAMP.ipynb`. For NMF the matrix is shifted to be non-negative. Outputs written to `output/01_model_building/05_pseudobulk/<dataset>/PCA/`, `.../NMF/`, `.../ICA/`.

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from pyprojroot.here import here
from sklearn.decomposition import PCA, NMF, FastICA

## Configuration

In [ ]:
DATASET  = "PBMC_Perez2022"
OUT_ROOT = "output/01_model_building/05_pseudobulk"

## Build models for each dataset

In [ ]:
import pickle

print(f"========== {DATASET} ==========")
from pyprojroot.here import here
from pathlib import Path
ds_dir = Path(here(OUT_ROOT)) / DATASET

# Load preprocessed data
norm_path = ds_dir / "norm.csv"
if not norm_path.exists():
    raise FileNotFoundError(f"{norm_path} not found — run 00_preprocess.ipynb first.")

norm = pd.read_csv(norm_path, index_col=0).astype(np.float32)
k    = int(pd.read_csv(ds_dir / "k.csv")["k"].iloc[0])
print(f"  norm shape: {norm.shape}  k={k}")

X = norm.T
gene_names   = norm.index.tolist()
sample_names = norm.columns.tolist()

# PCA
print("  Running PCA ...")
pca = PCA(n_components=k, svd_solver="auto", random_state=123)
W_pca = pca.fit_transform(X)
H_pca = pca.components_

pc_names = [f"PC{i+1}" for i in range(W_pca.shape[1])]
B_pca = pd.DataFrame(W_pca.T, index=pc_names, columns=sample_names)
Z_pca = pd.DataFrame(H_pca.T, index=gene_names, columns=pc_names)

pca_dir = ds_dir / "PCA"
pca_dir.mkdir(parents=True, exist_ok=True)
B_pca.to_csv(pca_dir / "B.csv")
Z_pca.to_csv(pca_dir / "Z.csv")
with open(pca_dir / "pca_model.pkl", "wb") as f: pickle.dump(pca, f)
print(f"  PCA saved -> {pca_dir}")

# ICA
print("  Running ICA ...")
ica = FastICA(n_components=k, random_state=123, max_iter=2000, tol=0.01)
W_ica = ica.fit_transform(X)
H_ica = ica.mixing_.T

ic_names = [f"IC{i+1}" for i in range(W_ica.shape[1])]
B_ica = pd.DataFrame(W_ica.T, index=ic_names, columns=sample_names)
Z_ica = pd.DataFrame(H_ica.T, index=gene_names, columns=ic_names)

ica_dir = ds_dir / "ICA"
ica_dir.mkdir(parents=True, exist_ok=True)
B_ica.to_csv(ica_dir / "B.csv")
Z_ica.to_csv(ica_dir / "Z.csv")
with open(ica_dir / "ica_model.pkl", "wb") as f: pickle.dump(ica, f)
print(f"  ICA saved -> {ica_dir}")

# NMF (shift norm to non-negative)
print("  Running NMF ...")
gene_min    = norm.min(axis=1)
norm_nonneg = norm.sub(gene_min, axis=0)
X_nmf = norm_nonneg.T

nmf = NMF(n_components=k, init="nndsvd", random_state=123, max_iter=1000)
W_nmf = nmf.fit_transform(X_nmf)
H_nmf = nmf.components_

lv_names = [f"LV{i+1}" for i in range(W_nmf.shape[1])]
B_nmf = pd.DataFrame(W_nmf.T, index=lv_names, columns=sample_names)
Z_nmf = pd.DataFrame(H_nmf.T, index=gene_names, columns=lv_names)

nmf_dir = ds_dir / "NMF"
nmf_dir.mkdir(parents=True, exist_ok=True)
B_nmf.to_csv(nmf_dir / "B.csv")
Z_nmf.to_csv(nmf_dir / "Z.csv")
with open(nmf_dir / "nmf_model.pkl", "wb") as f: pickle.dump(nmf, f)
print(f"  NMF saved -> {nmf_dir}")

print("\nDone.")